In [26]:
import sqlite3
import csv
import urllib.request
import os

In [27]:
import os

if os.path.exists("/content/library.db"):
    os.remove("/content/library.db")
    print("Old database removed.")
else:
    print("No old database found.")

No old database found.


In [43]:
BASE_URL = "https://raw.githubusercontent.com/jingwenr2/CIS3120-LibraryModule7/main/"

BOOK_URL = BASE_URL + "Book.csv"
MEMBER_URL = BASE_URL + "Member.csv"
LOAN_URL = BASE_URL + "Loan.csv"

BOOK_PATH = "/content/Book.csv"
MEMBER_PATH = "/content/Member.csv"
LOAN_PATH = "/content/Loan.csv"
DB_PATH = "/content/library.db"

In [29]:
for url, path in [(BOOK_URL, BOOK_PATH), (MEMBER_URL, MEMBER_PATH), (LOAN_URL, LOAN_PATH)]:
    urllib.request.urlretrieve(url, path)
    print(f"Downloaded: {path}")

Downloaded: /content/Book.csv
Downloaded: /content/Member.csv
Downloaded: /content/Loan.csv


In [44]:
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

conn.execute("""
CREATE TABLE IF NOT EXISTS Book (
    callNo TEXT NOT NULL,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    PRIMARY KEY (callNo)
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS Member (
    id INTEGER NOT NULL,
    firstname TEXT NOT NULL,
    lastName TEXT NOT NULL,
    PRIMARY KEY (id)
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS Loan (
    callNo TEXT NOT NULL,
    id INTEGER NOT NULL,
    dateBorrowed TEXT NOT NULL,
    dateReturned TEXT,
    dateDue TEXT NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id) REFERENCES Member(id)
);
""")

conn.commit()
print("Tables created.")

Tables created.


In [31]:
print(conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;").fetchall())

[('Book',), ('Loan',), ('Member',)]


In [32]:
with open(BOOK_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?);",
            (row["callNo"], row["title"], row["author"])
        )

conn.commit()
print("Book rows loaded:", conn.execute("SELECT COUNT(*) FROM Book;").fetchone()[0])

Book rows loaded: 11


In [33]:
with open(MEMBER_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?);",
            (int(row["id"]), row["firstname"], row["lastName"])
        )

conn.commit()
print("Member rows loaded:", conn.execute("SELECT COUNT(*) FROM Member;").fetchone()[0])

Member rows loaded: 4


In [34]:
with open(LOAN_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        date_returned = row["dateReturned"] if row["dateReturned"].strip() else None
        conn.execute("""
            INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)
            VALUES (?, ?, ?, ?, ?);
        """, (
            row["callNo"],
            int(row["id"]),
            row["dateBorrowed"],
            date_returned,
            row["dateDue"]
        ))

conn.commit()
print("Loan rows loaded:", conn.execute("SELECT COUNT(*) FROM Loan;").fetchone()[0])

Loan rows loaded: 4


In [35]:
print("Book:", conn.execute("SELECT COUNT(*) FROM Book;").fetchone()[0])
print("Member:", conn.execute("SELECT COUNT(*) FROM Member;").fetchone()[0])
print("Loan:", conn.execute("SELECT COUNT(*) FROM Loan;").fetchone()[0])

Book: 11
Member: 4
Loan: 4


## Query 1 - All Books

Retrieve all columns from the Book table, ordered alphabetically by author last name.

In [36]:
query1 = """
SELECT *
FROM Book
ORDER BY author;
"""

for row in conn.execute(query1):
    print(row)

('R 487 T35 1967', 'Medicine in medieval England.', 'Charles H Talbot')
('QA 76.9 D26H39 1996', 'Data model patterns : conventions of thought', 'David Hay')
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.9 D35C45 1999', "Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('R 141 E45 2006', 'Medieval medicine and the plague', 'Lynne Elliott')
('QA 76.9 D26H355 2008', 'Information modeling and relational databases', 'T A Halpin')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')


## Query 2 - Books Not Yet Returned

Retrieve the title of each book, and the first and last name of the member who borrowed it, for all loans where dateReturned is NULL.

In [37]:
query2 = """
SELECT Book.title, Member.firstname, Member.lastName
FROM Loan
JOIN Book ON Loan.callNo = Book.callNo
JOIN Member ON Loan.id = Member.id
WHERE Loan.dateReturned IS NULL;
"""

for row in conn.execute(query2):
    print(row)

("Joe Celko's SQL puzzles & answers", 'David', 'Martin')
('Medieval medicine and the plague', 'David', 'Martin')


## Query 3 - Loan History for a Specific Book

Retrieve the full loan history for the book with callNo R 141 E45 2006, including the member's full name, dateBorrowed, dateDue, and dateReturned. Order by dateBorrowed ascending.

In [38]:
query3 = """
SELECT Member.firstname, Member.lastName, Loan.dateBorrowed, Loan.dateDue, Loan.dateReturned
FROM Loan
JOIN Member ON Loan.id = Member.id
WHERE Loan.callNo = 'R 141 E45 2006'
ORDER BY Loan.dateBorrowed ASC;
"""

for row in conn.execute(query3):
    print(row)

('Betty', 'Freeman', '4/1/2014 0:00', '4/15/2014 0:00', '4/15/2014 0:00')
('David', 'Martin', '4/30/2014 0:00', '5/14/2014 0:00', None)


## Query 4 - Members Who Have Never Borrowed a Book

Retrieve the id, firstname, and lastName of every member who does not appear in the Loan table.

In [39]:
query4 = """
SELECT Member.id, Member.firstname, Member.lastName
FROM Member
LEFT JOIN Loan ON Member.id = Loan.id
WHERE Loan.id IS NULL;
"""

for row in conn.execute(query4):
    print(row)

(4, 'John', 'Martin')


## Query 5 - Count of Loans per Member

Retrieve each member's full name and the total number of loans they have made, including members with zero loans. Order by number of loans descending.

In [40]:
query5 = """
SELECT Member.firstname, Member.lastName, COUNT(Loan.id) AS loan_count
FROM Member
LEFT JOIN Loan ON Member.id = Loan.id
GROUP BY Member.id, Member.firstname, Member.lastName
ORDER BY loan_count DESC;
"""

for row in conn.execute(query5):
    print(row)

('David', 'Martin', 2)
('John', 'Smith', 1)
('Betty', 'Freeman', 1)
('John', 'Martin', 0)


## Query 6 - Free-Choice Analytical Query

**Business Question:** Which members currently have the most books still checked out?

**Why it is useful:** This helps the library identify which members are currently holding the most outstanding books. It can support follow-up, reminders, and a better understanding of active borrowing behavior.

In [41]:
query6 = """
SELECT Member.firstname, Member.lastName, COUNT(Loan.callNo) AS books_still_out
FROM Member
JOIN Loan ON Member.id = Loan.id
WHERE Loan.dateReturned IS NULL
GROUP BY Member.id, Member.firstname, Member.lastName
ORDER BY books_still_out DESC;
"""

for row in conn.execute(query6):
    print(row)

('David', 'Martin', 2)


## Summary

One data quality issue in this dataset is that some dateReturned values are blank, so they must be converted to NULL before loading into SQLite. Another observation is that the dates are stored as text instead of a true date type. One limitation of this dataset is that it is very small and does not include more realistic library details, such as fines, genres, or multiple copies of the same book. Because of that, it does not fully represent how a real public library system works.

In [42]:
conn.close()